In [14]:
import re
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.interpolate import CubicSpline

# 1. CONFIGURATION
INPUT_CSV      = "../../data/raw/dataset.csv"
SAMPLE_SUB_CSV = "../../data/raw/sandbox_solution.csv"
KAGGLE_OUT     = "../../submission_master.csv"
IV_CLIP_LO     = 0.02
IV_CLIP_HI     = 1.50

# 2. LOAD DATA
df = pd.read_csv(INPUT_CSV, parse_dates=["datetime"]).sort_values("datetime").reset_index(drop=True)
iv_cols = [c for c in df.columns if c not in ("datetime", "underlying_price")]
strike_map = {c: int(re.search(r"(\d+)(?:CE|PE)$", c).group(1)) for c in iv_cols}


# 3. PHASE 1: SPLINE INTERPOLATION (The Anchor)
print("Running Phase 1: Fitting Spline Baseline...")
df_baseline = df.copy()
strikes = np.array([strike_map[c] for c in iv_cols])

for idx, row in df.iterrows():
    iv_vals = row[iv_cols].values.astype(float)
    mask = ~np.isnan(iv_vals) & (iv_vals > 0)
    
    if mask.sum() >= 4:
        s_vals = strikes[mask]
        i_vals = iv_vals[mask]
        # Sort to ensure strictly increasing sequence for CubicSpline
        sort_idx = np.argsort(s_vals)
        cs = CubicSpline(s_vals[sort_idx], i_vals[sort_idx], bc_type='natural', extrapolate=True)
        
        predicted = cs(strikes)
        df_baseline.loc[idx, iv_cols] = np.where(mask, iv_vals, np.clip(predicted, IV_CLIP_LO, IV_CLIP_HI))

# 4. PHASE 2: ML RESIDUAL CORRECTION (The Corrector)
print("Running Phase 2: Training ML Residual Corrector...")
df_train_long = df.melt(id_vars=['datetime', 'underlying_price'], value_vars=iv_cols, var_name='contract', value_name='true_iv')
df_base_long  = df_baseline.melt(id_vars=['datetime'], value_vars=iv_cols, var_name='contract', value_name='spline_iv')

df_model = pd.merge(df_train_long, df_base_long, on=['datetime', 'contract'], how='inner').dropna(subset=['true_iv'])
df_model['target'] = df_model['true_iv'] - df_model['spline_iv']

# Feature Engineering
df_model['strike'] = df_model['contract'].str.extract(r'(\d+)').astype(float)
df_model['moneyness'] = np.log(df_model['strike'] / df_model['underlying_price'])
df_model['moneyness_sq'] = df_model['moneyness'] ** 2
df_model['moneyness_cb'] = df_model['moneyness'] ** 3

features = ['moneyness', 'moneyness_sq', 'moneyness_cb', 'spline_iv']
model = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.03, max_depth=6, tree_method='hist')
model.fit(df_model[features], df_model['target'])

# 5. INFERENCE: Apply to whole surface
print("Applying ML corrections...")
df_full = df_baseline.melt(id_vars=['datetime', 'underlying_price'], value_vars=iv_cols, var_name='contract', value_name='spline_iv')
df_full['strike'] = df_full['contract'].str.extract(r'(\d+)').astype(float)
df_full['moneyness'] = np.log(df_full['strike'] / df_full['underlying_price'])
df_full['moneyness_sq'] = df_full['moneyness'] ** 2
df_full['moneyness_cb'] = df_full['moneyness'] ** 3

df_full['datetime'] = pd.to_datetime(df_full['datetime'], dayfirst=True)

df_full['id'] = df_full['datetime'].dt.strftime('%d-%m-%Y %H:%M') + '||' + df_full['contract']
df_full['predicted_iv'] = df_full['spline_iv'] + model.predict(df_full[features])
df_full['predicted_iv'] = df_full['predicted_iv'].clip(IV_CLIP_LO, IV_CLIP_HI)


Running Phase 1: Fitting Spline Baseline...
Running Phase 2: Training ML Residual Corrector...
Applying ML corrections...


In [15]:

# 6. KAGGLE FORMATTING
print("Formatting Kaggle submission...")
sub_template = pd.read_csv(SAMPLE_SUB_CSV)
final_sub = pd.merge(sub_template[['id']], df_full[['id', 'predicted_iv']], on='id', how='left')
final_sub = final_sub.rename(columns={'predicted_iv': 'value'}).fillna(0.20).round(6)

final_sub.to_csv(KAGGLE_OUT, index=False)
print(f"SUCCESS! Master Pipeline complete. Output: {KAGGLE_OUT}")

Formatting Kaggle submission...
SUCCESS! Master Pipeline complete. Output: ../../submission_master.csv
